# Segment 3 — LiDAR Semantic Segmentation

Prototype pipeline: **LiDAR (x, y, z, intensity) → point-wise class + confidence**.

Classes: Road, Non-drivable Terrain, Vehicle, Pedestrian, Building/Wall, Pole, Vegetation, Unknown.

In [13]:
!pip install numpy torch open3d matplotlib -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open3d as o3d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [15]:
CLASS_NAMES = {
    0: 'Road',
    1: 'Non-drivable Terrain',
    2: 'Vehicle',
    3: 'Pedestrian',
    4: 'Building/Wall',
    5: 'Pole',
    6: 'Vegetation',
    7: 'Unknown'
}

NUM_CLASSES = len(CLASS_NAMES)
for k, v in CLASS_NAMES.items(): print(k, '→', v)

0 → Road
1 → Non-drivable Terrain
2 → Vehicle
3 → Pedestrian
4 → Building/Wall
5 → Pole
6 → Vegetation
7 → Unknown


## 1. Load LiDAR `.bin` file

In [16]:
def load_lidar_bin(path):
    data = np.fromfile(path, dtype=np.float32)
    if data.size % 4 != 0:
        raise ValueError('Expected float32 x,y,z,intensity values.')
    return data.reshape(-1, 4)

LIDAR_PATH = 'data/000000.bin'

if os.path.exists(LIDAR_PATH):
    points = load_lidar_bin(LIDAR_PATH)
    print('Point cloud shape:', points.shape)
    print(points[:5])
else:
    print('File not found:', LIDAR_PATH)
    print('Put your .bin file inside data/ and update LIDAR_PATH.')

File not found: data/000000.bin
Put your .bin file inside data/ and update LIDAR_PATH.


In [17]:
def normalize_points(points):
    xyz = points[:, :3].astype(np.float32)
    intensity = points[:, 3:4].astype(np.float32)
    xyz = xyz - np.mean(xyz, axis=0)
    scale = np.max(np.linalg.norm(xyz, axis=1))
    if scale > 0:
        xyz = xyz / scale
    return np.concatenate([xyz, intensity], axis=1).astype(np.float32)

## 2. PointNet-style segmentation model

In [18]:
class PointNetSegmentation(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(4, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.conv4 = nn.Conv1d(384, 256, 1)
        self.conv5 = nn.Conv1d(256, 128, 1)
        self.conv6 = nn.Conv1d(128, num_classes, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.bn4 = nn.BatchNorm1d(256)
        self.bn5 = nn.BatchNorm1d(128)

    def forward(self, x):
        x = x.transpose(2, 1)
        x1 = F.relu(self.bn1(self.conv1(x)))
        x2 = F.relu(self.bn2(self.conv2(x1)))
        x3 = F.relu(self.bn3(self.conv3(x2)))
        global_feature = torch.max(x3, dim=2, keepdim=True)[0]
        global_feature = global_feature.repeat(1, 1, x.shape[2])
        features = torch.cat([x2, global_feature], dim=1)
        features = F.relu(self.bn4(self.conv4(features)))
        features = F.relu(self.bn5(self.conv5(features)))
        return self.conv6(features).transpose(2, 1)

model = PointNetSegmentation(NUM_CLASSES).to(device)
print('Model ready.')

Model ready.


## 3. Labeled dataset loader

In [19]:
class LidarDataset(Dataset):
    def __init__(self, point_files, label_files, num_points=4096):
        if len(point_files) != len(label_files):
            raise ValueError('Point and label file counts must match.')
        self.point_files = point_files
        self.label_files = label_files
        self.num_points = num_points

    def __len__(self): return len(self.point_files)

    def __getitem__(self, idx):
        points = load_lidar_bin(self.point_files[idx])
        labels = np.fromfile(self.label_files[idx], dtype=np.int32)
        if len(points) != len(labels):
            raise ValueError('Point/label count mismatch.')
        indices = np.random.choice(len(points), self.num_points,
                                   replace=len(points) < self.num_points)
        return (torch.tensor(normalize_points(points[indices]), dtype=torch.float32),
                torch.tensor(labels[indices], dtype=torch.long))

## 4. Training

Set `RUN_TRAINING = True` only after adding correctly labeled training data.

In [20]:
POINT_FILES = ['dataset/points/000000.bin', 'dataset/points/000001.bin']
LABEL_FILES = ['dataset/labels/000000.label', 'dataset/labels/000001.label']
RUN_TRAINING = False

if RUN_TRAINING:
    dataset = LidarDataset(POINT_FILES, LABEL_FILES, num_points=4096)
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    EPOCHS = 20

    for epoch in range(EPOCHS):
        model.train(); total_loss = 0.0
        for batch_points, batch_labels in loader:
            batch_points, batch_labels = batch_points.to(device), batch_labels.to(device)
            output = model(batch_points)
            loss = criterion(output.reshape(-1, NUM_CLASSES), batch_labels.reshape(-1))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f}')

    torch.save(model.state_dict(), 'lidar_segmentation_model.pth')
    print('Saved lidar_segmentation_model.pth')
else:
    print('Training OFF — add labeled data first.')

Training OFF — add labeled data first.


## 5. Load trained model

In [21]:
MODEL_PATH = 'lidar_segmentation_model.pth'
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print('Trained model loaded.')
else:
    print('No trained model found.')

No trained model found.


## 6. Predict class + confidence

In [22]:
def predict_lidar(model, points):
    model.eval()
    x = torch.tensor(normalize_points(points), dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(x)
        probabilities = torch.softmax(output, dim=-1)
        confidence, predictions = torch.max(probabilities, dim=-1)
    return predictions.cpu().numpy()[0], confidence.cpu().numpy()[0]

if 'points' in globals():
    predictions, confidence = predict_lidar(model, points)
   
    for i in range(min(20, len(points))):
        print(f'Point {i:5d}: {CLASS_NAMES[int(predictions[i])]:20s} | confidence = {confidence[i]:.3f}')

In [23]:
if 'points' in globals() and 'predictions' in globals():
    result = np.column_stack([points, predictions, confidence])
    np.save('frame_predictions.npy', result)
    print('Saved: frame_predictions.npy')
    print('Columns: x, y, z, intensity, class_id, confidence')

## 7. Visualize segmentation

In [24]:
def visualize_prediction(points, predictions):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points[:, :3])
    class_colors = {
        0:[0.2,0.8,0.2], 1:[0.7,0.7,0.2], 2:[0.9,0.1,0.1],
        3:[0.1,0.1,0.9], 4:[0.6,0.4,0.2], 5:[0.8,0.2,0.8],
        6:[0.1,0.6,0.1], 7:[0.5,0.5,0.5]
    }
    colors = np.zeros((len(points), 3))
    for class_id, color in class_colors.items():
        colors[predictions == class_id] = color
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd], window_name='LiDAR Semantic Segmentation')

if 'points' in globals() and 'predictions' in globals():
    visualize_prediction(points, predictions)

## Segment 3 target

**Input:** `(x, y, z, intensity)`

**Output:** `point → class → confidence`

The resulting `frame_predictions.npy` is the handoff to Segment 4, where labeled points are grouped into actual objects.